# Google Ngram Frequency Lookup — Figurative Language Dataset

**Fully automated.** This notebook:
1. Reads all idioms from `figurative_aut_orig.json` 
2. Queries the **Google Ngram Viewer API** for each idiom 
3. Saves the **average proportion** for each idiom into `fig_ngram_freq.json`



In [1]:
import json
import os
import time
import requests

print("✓ All libraries loaded")


✓ All libraries loaded


## Configuration

All settings in one place. No need to change anything unless you want a different year range.


In [ ]:
INPUT_FILE  = "figurative_aut_orig.json"
OUTPUT_FILE = "fig_ngram_freq.json"

YEAR_START  = 2000
YEAR_END    = 2019
CORPUS      = "en-2019"   # Google Ngram English corpus (most complete)
SMOOTHING   = 0           # No smoothing — raw proportions
DELAY       = 2           # Seconds to wait between requests

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Could not find: {INPUT_FILE}\n"
        "Make sure this notebook is inside the same folder as 'figurative_aut_orig.json'."
    )

print(f"Input file   : {INPUT_FILE}")
print(f"Output file  : {OUTPUT_FILE}")
print(f"Year range   : {YEAR_START}–{YEAR_END}")
print(f"Corpus       : {CORPUS}")


✓ Input file   : figurative_aut_orig.json
✓ Output file  : fig_ngram_freq.json
✓ Year range   : 2000–2019
✓ Corpus       : en-2019


## Load Idioms

Extracts the `fig` field from every entry, preserving original order.


In [ ]:
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

fig_list = [entry["fig"].strip() for entry in data if entry.get("fig", "").strip()]

print(f"Total idioms loaded: {len(fig_list)}")
print()
print("First 5 (order check):")
for i, fig in enumerate(fig_list[:5], 1):
    print(f"  [{i}] {fig}")


✓ Total idioms loaded: 100

First 5 (order check):
  [1] wearing thin
  [2] the brain behind the operation
  [3] irons in the fire
  [4] heart of gold
  [5] has wings


## Ngram Query Function

This function queries the Google Ngram API for one idiom and returns the **average proportion** across 2000–2022.  
- If the idiom is found, returns a float (e.g. `0.00000342`)  
- If not found in the corpus, returns `0.0`  
- If the API fails, returns `null`


In [16]:
def query_ngram(phrase: str) -> float | None:
    """
    Query Google Ngram API for a phrase and return its average proportion
    over the specified year range. Returns None on error.
    """
    url = "https://books.google.com/ngrams/json"
    params = {
        "content"    : phrase,
        "year_start" : YEAR_START,
        "year_end"   : YEAR_END,
        "corpus"     : CORPUS,
        "smoothing"  : SMOOTHING,
    }
    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        results = response.json()

        if not results:
            return 0.0

        timeseries = results[0].get("timeseries", [])
        if not timeseries:
            return 0.0

        avg_proportion = sum(timeseries) / len(timeseries)
        return round(avg_proportion, 10)

    except requests.exceptions.RequestException as e:
        print(f"    [!] Request error: {e}")
        return None
    except (ValueError, KeyError, IndexError) as e:
        print(f"    [!] Parse error: {e}")
        return None

print("query_ngram() function ready")


query_ngram() function ready


## Run All Queries 




In [15]:
results_list = []

print("=" * 70)
print(f"  Querying Google Ngram API for {len(fig_list)} idioms...")
print(f"  Year range: {YEAR_START}–{YEAR_END}  |  Delay: {DELAY}s between requests")
print("=" * 70)
print()

for i, fig in enumerate(fig_list, start=1):
    print(f"[{i:>3}/{len(fig_list)}]  {fig}")
    
    proportion = query_ngram(fig)
    
    if proportion is None:
        status = "ERROR — saved as null"
    elif proportion == 0.0:
        status = "not found in corpus (0.0)"
    else:
        status = f"{proportion:.10f}"
    
    print(f"          proportion : {status}")
    print()

    results_list.append({
        "fig"        : fig,
        "proportion" : proportion
    })

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(results_list, f, indent=2, ensure_ascii=False)

    if i < len(fig_list):
        time.sleep(DELAY)

print("=" * 70)
print(f"Done. All {len(results_list)} entries saved to: {OUTPUT_FILE}")
print("=" * 70)


  Querying Google Ngram API for 100 idioms...
  Year range: 2000–2019  |  Delay: 2s between requests

[  1/100]  wearing thin
          proportion : 0.0000001092

[  2/100]  the brain behind the operation
          proportion : 0.0000000002

[  3/100]  irons in the fire
          proportion : 0.0000000370

[  4/100]  heart of gold
          proportion : 0.0000001368

[  5/100]  has wings
          proportion : 0.0000000439

[  6/100]  stirred the pot
          proportion : 0.0000000196

[  7/100]  skating on thin ice
          proportion : 0.0000000188

[  8/100]  walking encyclopedia
          proportion : 0.0000000193

[  9/100]  hit a wall
          proportion : 0.0000000721

[ 10/100]  cut like a knife
          proportion : 0.0000000207

[ 11/100]  tension in the room
          proportion : 0.0000001093

[ 12/100]  in hot water
          proportion : 0.0000003771

[ 13/100]  jumped through hoops
          proportion : 0.0000000037

[ 14/100]  tip of the iceberg
          proportio

##  Summary & Verification

Check final results how many were found, how many were missing, any errors.


In [17]:
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    saved = json.load(f)

found   = [e for e in saved if e["proportion"] and e["proportion"] > 0.0]
missing = [e for e in saved if e["proportion"] == 0.0]
errors  = [e for e in saved if e["proportion"] is None]

print("=" * 70)
print(f"  SUMMARY — {OUTPUT_FILE}")
print("=" * 70)
print(f"  Total entries  : {len(saved)}")
print(f"  Found in corpus: {len(found)}")
print(f"  Not found (0.0): {len(missing)}")
print(f"  Errors (null)  : {len(errors)}")
print()

if missing:
    print("Idioms NOT found in Ngram corpus:")
    for e in missing:
        print(f"  - {e['fig']}")
    print()

if errors:
    print("Idioms with errors (API failed):")
    for e in errors:
        print(f"  - {e['fig']}")
    print()

print("-" * 70)
print("All results:")
for i, entry in enumerate(saved, 1):
    prop = entry['proportion']
    prop_str = f"{prop:.10f}" if prop else str(prop)
    print(f"[{i:>3}]  {entry['fig']:<45}  {prop_str}")


  SUMMARY — fig_ngram_freq.json
  Total entries  : 100
  Found in corpus: 89
  Not found (0.0): 11
  Errors (null)  : 0

Idioms NOT found in Ngram corpus:
  - pulled a rabbit out of his hat
  - burning the candle at both ends
  - the cat out of the bag
  - get our ducks in a row
  - hit the nail on the head
  - biting off more than he can chew
  - took the wind out of my sails
  - apple of her father's eye
  - strike while the iron is hot
  - all his eggs in one basket
  - a mountain out of a molehill

----------------------------------------------------------------------
All results:
[  1]  wearing thin                                   0.0000001092
[  2]  the brain behind the operation                 0.0000000002
[  3]  irons in the fire                              0.0000000370
[  4]  heart of gold                                  0.0000001368
[  5]  has wings                                      0.0000000439
[  6]  stirred the pot                                0.0000000196
[  7] 

## Retry Any Errors

If any entries came back as `null` due to API/network issues, this cell retries just those.


In [18]:
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    saved = json.load(f)

errors = [e for e in saved if e["proportion"] is None]

if not errors:
    print("No errors to retry — all entries are complete!")
else:
    print(f"Retrying {len(errors)} failed entries...\n")
    
    for entry in saved:
        if entry["proportion"] is None:
            fig = entry["fig"]
            print(f"  Retrying: {fig}")
            proportion = query_ngram(fig)
            entry["proportion"] = proportion
            status = f"{proportion:.10f}" if proportion else str(proportion)
            print(f"  Result  : {status}\n")
            time.sleep(DELAY)
    
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(saved, f, indent=2, ensure_ascii=False)
    
    print(f" Retries complete. File updated: {OUTPUT_FILE}")


No errors to retry — all entries are complete!
